# Exp-A/B/C/DXY/Spot Feature Comparison Experiment (Updated)

This notebook compares several feature engineering ideas against the baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.append("..")

from src.processing import load_and_clean_data
from src.features import generate_features, frac_diff
from src.pooling import pool_boj_data
from src.modeling import walk_forward_validation, summarize_ic

# Load data
df_raw = load_and_clean_data("../data/BOJ_data.xlsx", "../data/BOJ_meeting_history.csv")
print(f"Data loaded. Shape: {df_raw.shape}")
print(f"Columns in df_raw: {df_raw.columns.tolist()}")

## Baseline (42 features)

In [ ]:
# 1. Baseline
df_feat_base = generate_features(df_raw)  # Default: all flags False
df_pooled_base = pool_boj_data(df_feat_base)

res_3d_base = walk_forward_validation(df_pooled_base, "Target_3d_norm", "2024-01-01")
res_5d_base = walk_forward_validation(df_pooled_base, "Target_5d_norm", "2024-01-01")

ic_3d_base = summarize_ic(res_3d_base)
ic_5d_base = summarize_ic(res_5d_base)

print("Baseline IC (3d):", round(ic_3d_base["ic_all"], 4))
print("Baseline IC (5d):", round(ic_5d_base["ic_all"], 4))

## Exp-A: Weekday Cyclic Encoding

In [ ]:
# Exp-A
df_feat_a = generate_features(df_raw, add_weekday_cyclic=True)
df_pooled_a = pool_boj_data(df_feat_a)

res_3d_a = walk_forward_validation(df_pooled_a, "Target_3d_norm", "2024-01-01")
res_5d_a = walk_forward_validation(df_pooled_a, "Target_5d_norm", "2024-01-01")

ic_3d_a = summarize_ic(res_3d_a)
ic_5d_a = summarize_ic(res_5d_a)

print("Exp-A IC (3d):", round(ic_3d_a["ic_all"], 4))
print("Exp-A IC (5d):", round(ic_5d_a["ic_all"], 4))
print("\nExp-A 3d ic_by_fold:")
print(ic_3d_a["ic_by_fold"])
print("\nExp-A 5d ic_by_fold:")
print(ic_5d_a["ic_by_fold"])

## Exp-DXY: Add DXY_frac_diff

In [ ]:
def generate_features_with_dxy(df, d=0.4, window=50):
    feat_df = generate_features(df, d=d, window=window)
    if "DXY" in feat_df.columns:
        feat_df["DXY_frac_diff"] = frac_diff(feat_df["DXY"], d=d, window=window)
        print(f"DXY found. DXY_frac_diff head:\n{feat_df["DXY_frac_diff"].dropna().head()}")
    else:
        print("DXY NOT found in feat_df columns!")
    return feat_df

df_feat_dxy = generate_features_with_dxy(df_raw)
df_pooled_dxy = pool_boj_data(df_feat_dxy)

if "DXY_frac_diff" not in df_pooled_dxy.columns:
    print("DXY_frac_diff missing from pooled, merging manually...")
    dxy_map = df_feat_dxy[["Date", "DXY_frac_diff"]].drop_duplicates()
    df_pooled_dxy = df_pooled_dxy.merge(dxy_map, on="Date", how="left")

print(f"Is DXY_frac_diff in df_pooled_dxy? {"DXY_frac_diff" in df_pooled_dxy.columns}")
if "DXY_frac_diff" in df_pooled_dxy.columns:
    print(f"Non-NaN DXY_frac_diff count in pooled: {df_pooled_dxy["DXY_frac_diff"].notnull().sum()}")

res_3d_dxy = walk_forward_validation(df_pooled_dxy, "Target_3d_norm", "2024-01-01")
res_5d_dxy = walk_forward_validation(df_pooled_dxy, "Target_5d_norm", "2024-01-01")

ic_3d_dxy = summarize_ic(res_3d_dxy)
ic_5d_dxy = summarize_ic(res_5d_dxy)
print("Exp-DXY IC (3d):", round(ic_3d_dxy["ic_all"], 4))
print("Exp-DXY IC (5d):", round(ic_5d_dxy["ic_all"], 4))

## Summary of Results

In [ ]:
results = {
    "Baseline": {
        "3d_ic_all": ic_3d_base["ic_all"],
        "3d_ic_recent": ic_3d_base["ic_recent"],
        "5d_ic_all": ic_5d_base["ic_all"],
        "5d_ic_recent": ic_5d_base["ic_recent"],
    },
    "Exp-A (+weekday)": {
        "3d_ic_all": ic_3d_a["ic_all"],
        "3d_ic_recent": ic_3d_a["ic_recent"],
        "5d_ic_all": ic_5d_a["ic_all"],
        "5d_ic_recent": ic_5d_a["ic_recent"],
    },
    "Exp-DXY (+DXY_fd)": {
        "3d_ic_all": ic_3d_dxy["ic_all"],
        "3d_ic_recent": ic_3d_dxy["ic_recent"],
        "5d_ic_all": ic_5d_dxy["ic_all"],
        "5d_ic_recent": ic_5d_dxy["ic_recent"],
    },
}

df_results = pd.DataFrame(results).T
df_results["3d_delta"] = df_results["3d_ic_all"] - df_results.loc["Baseline", "3d_ic_all"]
df_results["5d_delta"] = df_results["5d_ic_all"] - df_results.loc["Baseline", "5d_ic_all"]
print(df_results.round(4).to_string())